In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Specifică observabile în baza Pauli

<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
```
</details>
În mecanica cuantică, observabilele corespund proprietăților fizice care pot fi măsurate.
Când consideri un sistem de spinuri, de exemplu, ai putea fi interesat să măsori energia sistemului sau să obții informații despre alinierea spinurilor, cum ar fi magnetizarea sau corelațiile dintre spinuri.

Pentru a măsura un observabil $n$-Qubit $O$ pe un calculator cuantic, trebuie să îl reprezinți ca o sumă de produse tensoriale ale operatorilor Pauli, adică

$$
O = \sum_{k=1}^K \alpha_k P_k,~~ P_k \in {I, X, Y, Z}^{\otimes n},~~ \alpha_k \in \mathbb{R},
$$

unde

$$
I = \begin{pmatrix}
1 & 0 \\ 0 & 1
\end{pmatrix}
~~
X = \begin{pmatrix}
0 & 1 \\ 1 & 0
\end{pmatrix}
~~
Y = \begin{pmatrix}
0 & -i \\ i & 0
\end{pmatrix}
~~
Z = \begin{pmatrix}
1 & 0 \\ 0 & -1
\end{pmatrix}
$$

și folosești faptul că un observabil este hermitian, adică $O^\dagger = O$. Dacă $O$ nu este hermitian, poate fi totuși descompus ca o sumă de Pauli, dar coeficientul $\alpha_k$ devine complex.

În multe cazuri, observabilul este specificat în mod natural în această reprezentare după maparea sistemului de interes la Qubiți.
De exemplu, un sistem de spin-1/2 poate fi mapat la un Hamiltonian Ising

$$
H = \sum_{\langle i, j\rangle} Z_i Z_j - \sum_{i=1}^n X_i,
$$

unde indicii $\langle i, j\rangle$ parcurg spinurile care interacționează, iar spinurile sunt supuse unui câmp transversal în $X$.
Indicele subscris indică pe ce Qubit acționează operatorul Pauli, adică $X_i$ aplică un operator $X$ pe Qubit-ul $i$ și lasă restul neschimbat.

În Qiskit SDK, acest Hamiltonian poate fi construit cu următorul cod.

In [1]:
from qiskit.quantum_info import SparsePauliOp

# define the number of qubits
n = 12

# define the single Pauli terms as ("Paulis", [indices], coefficient)
interactions = [
    ("ZZ", [i, i + 1], 1) for i in range(n - 1)
]  # we assume spins on a 1D line
field = [("X", [i], -1) for i in range(n)]

# build the operator
hamiltonian = SparsePauliOp.from_sparse_list(
    interactions + field, num_qubits=n
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIIIIZZ', 'IIIIIIIIIZZI', 'IIIIIIIIZZII', 'IIIIIIIZZIII', 'IIIIIIZZIIII', 'IIIIIZZIIIII', 'IIIIZZIIIIII', 'IIIZZIIIIIII', 'IIZZIIIIIIII', 'IZZIIIIIIIII', 'ZZIIIIIIIIII', 'IIIIIIIIIIIX', 'IIIIIIIIIIXI', 'IIIIIIIIIXII', 'IIIIIIIIXIII', 'IIIIIIIXIIII', 'IIIIIIXIIIII', 'IIIIIXIIIIII', 'IIIIXIIIIIII', 'IIIXIIIIIIII', 'IIXIIIIIIIII', 'IXIIIIIIIIII', 'XIIIIIIIIIII'],
              coeffs=[ 1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,
  1.+0.j,  1.+0.j,  1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j,
 -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j])


If we would like to measure the energy the observable is the Hamiltonian itself. Alternatively, we could be
interested in measuring system properties like the average magnetization by counting the number of spins
aligned in the $Z$-direction with the observable

$$
O = \frac{1}{n} \sum_{i=1} Z_i
$$

For observables that are not given in terms of Pauli operators but in a matrix form, we first have to reformulate them
in the Pauli basis in order to evaluate them on a quantum computer.
We are always able to find such a representation as the Pauli matrices form a basis for the Hermitian $2^n \times 2^n$ matrices.
We expand the observable $O$ as

$$
O = \sum_{P \in \{I, X, Y, Z\}^{\otimes n}} \mathrm{Tr}(O P) P,
$$

where the sum runs over all possible $n$-qubit Pauli terms and $\mathrm{Tr}(\cdot)$ is the trace of a matrix, which acts as inner product.
You can implement this decomposition  from a matrix to Pauli terms using the `SparsePauliOp.from_operator` method, like so:

In [2]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp

matrix = np.array(
    [[-1, 0, 0.5, -1], [0, 1, 1, 0.5], [0.5, 1, -1, 0], [-1, 0.5, 0, 1]]
)

observable = SparsePauliOp.from_operator(matrix)
print(observable)

SparsePauliOp(['IZ', 'XI', 'YY'],
              coeffs=[-1. +0.j,  0.5+0.j,  1. -0.j])


Dacă dorim să măsurăm energia, observabilul este chiar Hamiltonianul. Alternativ, am putea fi
interesați să măsurăm proprietăți ale sistemului, cum ar fi magnetizarea medie, numărând spinurile
aliniate în direcția $Z$ cu observabilul

$$
O = \frac{1}{n} \sum_{i=1} Z_i
$$

Pentru observabilele care nu sunt date în termeni de operatori Pauli, ci sub formă de matrice, trebuie mai întâi să le reformulăm
în baza Pauli pentru a le evalua pe un calculator cuantic.
Putem întotdeauna găsi o astfel de reprezentare, deoarece matricele Pauli formează o bază pentru matricele hermitiene $2^n \times 2^n$.
Expandăm observabilul $O$ astfel:

$$
O = \sum_{P \in {I, X, Y, Z}^{\otimes n}} \mathrm{Tr}(O P) P,
$$

unde suma parcurge toți termenii Pauli posibili cu $n$ Qubiți, iar $\mathrm{Tr}(\cdot)$ este urma unei matrice, care acționează ca produs interior.
Poți implementa această descompunere de la o matrice la termeni Pauli folosind metoda `SparsePauliOp.from_operator`, astfel:

In [3]:
from qiskit.circuit import QuantumCircuit

# create a circuit, where we would like to measure
# q0 in the X basis, q1 in the Y basis and q2 in the Z basis
circuit = QuantumCircuit(3)
circuit.ry(0.8, 0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.barrier()

# diagonalize X with the Hadamard gate
circuit.h(0)

# diagonalize Y with Hadamard as S^\dagger
circuit.sdg(1)
circuit.h(1)

# the Z basis is the default, no action required here

# measure all qubits
circuit.measure_all()
circuit.draw("mpl")

<Image src="../docs/images/guides/specify-observables-pauli/extracted-outputs/ce4b1984-ebe0-44f6-a78c-d67b9e9bb361-0.svg" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
  -  Read the [SparsePauliOp API](/docs/api/qiskit/qiskit.quantum_info.SparsePauliOp#sparsepauliop) reference.
</Admonition>